# Solving surface PDEs with neural-CPM demo

We walk through:
   * Loading the surface features.
   * Building the **narrow band** around the surface.
   * **Space-partitioning** the band into overlapping local regions.
   * Running the **neural network** to obtain attention weight matrices.
   * Apply the neural extension by aggregating the local updates with neural weights and rotation invariance.
   * Projecting the band solution back onto the surface with **RBF** interp.

## 0. Setup


In [ ]:
%load_ext autoreload
%autoreload 2
import os, time, pickle, pathlib
import numpy as np
import torch
import matplotlib.pyplot as plt
import plotly.io as pio
from scipy.spatial import KDTree
from scipy.sparse.linalg import spsolve
from tqdm import tqdm

# ---- single device knob -------------------------------------------------
DEVICE = "cpu"        # change to "mps" or "cuda" if available
USE_CACHE = True      # cache the (slow) neural-weight retrieval to disk

# ---- plotly renderer ----------------------------------------------------
# Comment / uncomment to force a specific renderer. Default 'None' lets
# plotly auto-pick; this works in most local Jupyter/VS Code sessions.
# pio.renderers.default = "vscode"               # for VS Code notebooks
# pio.renderers.default = "notebook_connected"   # for classic Jupyter
# pio.renderers.default = "jupyterlab"           # for JupyterLab
# pio.renderers.default = "browser"              # pops a browser tab

torch.manual_seed(0)
np.random.seed(0)
print(f"torch {torch.__version__}  device = {DEVICE}")
print(f"plotly renderer = {pio.renderers.default!r}")

In [ ]:
from utils import (
    define_band_points,
    Laplacian_matrix,
    retrieve_neural_weights,
    neural_extension,
    precompute_rbf_data,
    interpolate_from_precomputed,
    plot_3d_point_cloud,
)
from utils import ncpm_viz as viz
from model import SurfNO_weights_only

## 1. Closest-Point Method

CPM solves a PDE that lives on a surface $S$ by:

* sampling $u$ on a thin **band** of voxels around $S$,
* repeatedly **extending** $u$ from $S$ into the band along surface normals
  (so $u$ is constant in the normal direction),
* and applying a standard 3-D differential operator (here the 7-point
  Laplacian) on the band.
  
Run the interactive HTML demo with python -m http.server 8000, then open http://localhost:8000/web/index.html

## 2. Neural CPM on the Apple

### 2.0 Loading the surface features

`Apple_surface_feature.npy` is an `(N, 12)` array per surface point (normalized):

| columns | meaning |
|--------:|---------|
| 0 : 3   | xyz position |
| 3 : 6   | outward unit normal |
| 6 : 9   | principal tangent $t_1$ |
| 9 : 12  | principal tangent $t_2$ |

In [ ]:
surface_feature_TP = np.load("data/Apple_surface_feature.npy")
surface_points     = surface_feature_TP[:, :3]
surface_normals    = surface_feature_TP[:, 3:6]
Tree_surface_points = KDTree(surface_points)

print("surface point cloud:", surface_points.shape, " bbox:", surface_points.min(0).round(3), "->", surface_points.max(0).round(3))

In [ ]:
viz.plot_surface_with_normals(surface_points, surface_normals,
                              subsample=80,
                              title="Apple surface with subsampled normals")
plt.show()

### 2.1 Narrow band around the surface

We discretise a thin shell of voxels around $S$. The function
`define_band_points` builds an axis-aligned grid with spacing `delta_x`,
keeps voxels closer than `dist_to_surface` to $S, and additionally returns
a Boolean `mask_threshold` flagging band points *far enough* from $S$ to
serve as seeds for the space-partition step.

In [ ]:
delta_x         = 0.05 # 0.05
dist_to_surface = 0.2 # 0.2
threshold       = 0.8 * dist_to_surface

band_points, mask_threshold = define_band_points(
    delta_x, surface_points, Tree_surface_points,
    dist_to_surface, threshold)
Tree_band_points = KDTree(band_points)

print(f"band points: {band_points.shape[0]},"
      f" seed points (mask_threshold=True): {mask_threshold.sum()}")

In [ ]:
# plot the band points, you can experiment with different parameters
viz.plot_band_overlay(surface_points, band_points, title="Surface + narrow band").show()

In [ ]:
# 2-D slice of the band 
viz.plot_band_slice(band_points, axis=2,
                    level=float(np.median(band_points[:, 2])),
                    tol=delta_x * 0.6,
                    title="Horizontal slice through the band")
plt.show()

### 2.2 Space partition into local regions

`space_partition` greedily covers the entire band with overlapping local
regions of fixed size `local_size`. Each region is anchored at a surface
point (its `central` point) and holds the `local_size` nearest band points.

In [ ]:
from utils.space_partition import space_partition

local_size = 400
(all_local_band_indexes,
 all_large_surface_masks,
 all_distances_to_central,
 indexes_central_points) = space_partition(
    surface_points, band_points,
    Tree_surface_points, Tree_band_points,
    local_size, mask_threshold)

print(f"#regions: {len(all_local_band_indexes)}    "
      f"avg band coverage per region: {local_size}")

In [ ]:
# Highlight one region
r = 300
viz.plot_local_region(band_points,
                      all_local_band_indexes[r],
                      central_pt=surface_points[indexes_central_points[r]],
                      surface_pts=surface_points,
                      title=f"Local region #{r} of {len(all_local_band_indexes)}")
plt.show()

### 2.3 Retrieving the neural-operator weights

Our model is loaded with the pretrained weights and, for every local region, produces an `(N, N)` attention matrix that acts as a learned smoothing operator on band functions. We compute these once and reuse them for **every** PDE solve below.

In [ ]:
cache_path = pathlib.Path("cache/neural_weights_2.pt")
cache_path.parent.mkdir(exist_ok=True)

# use the cached weigths first to see the visualization and output, then try it yourselves with no cache
if USE_CACHE and cache_path.exists():
    blob = torch.load(cache_path, map_location=DEVICE, weights_only=False)
    neural_weights           = [w.to(DEVICE) for w in blob["neural_weights"]]
    all_distances_to_central = blob["all_distances_to_central"]
    all_local_band_indexes   = blob["all_local_band_indexes"]
    print("loaded cached neural weights")
else:
    model = SurfNO_weights_only().to(DEVICE)
    model.load_state_dict(torch.load(
        "data/SurfNO_pretrained_weights.pth",
        map_location=DEVICE, weights_only=True))
    model.eval()
    t0 = time.time()
    with torch.no_grad():
        (neural_weights,
         all_distances_to_central,
         all_local_band_indexes) = retrieve_neural_weights(
            surface_points, band_points, local_size, model,
            Tree_band_points, mask_threshold,
            surface_feature_TP=surface_feature_TP)
    print(f"retrieve_neural_weights took {time.time()-t0:.1f} s")

    torch.save({
        "neural_weights":           [w.detach().cpu() for w in neural_weights],
        "all_distances_to_central": all_distances_to_central,
        "all_local_band_indexes":   all_local_band_indexes,
    }, cache_path)
    print(f"cached → {cache_path}")

In [ ]:
# Visualise the attention matrix of one local region
region = 0
alpha = neural_weights[0][region, 0].detach().cpu().numpy()   # (N, N)
viz.plot_attention(alpha, title=f"Attention for region {region}  (N=400)")
plt.show()
# Each row of the attention matrix tells you: to update u at this band point, how much should we weight u at every other band point of the same region.

### 2.4 Aggregating and neural extension 

A given band point may belong to several regions (overlap). neural extension applies the neural attention region-by-region (and rotation-by-rotation), then combines the predictions per band point with a distance-weighted
softmax.

Let us run it on a probe field $u(x) = \sin(\omega x)\sin(\omega y)\sin(\omega z)$.

In [ ]:
omega = 10
probe = (np.sin(omega*band_points[:,0]) * np.sin(omega*band_points[:,1]) * np.sin(omega*band_points[:,2]))
probe -= probe.mean()

probe_smoothed = neural_extension(neural_weights, probe, all_local_band_indexes, all_distances_to_central)[0]

fig = plt.figure(figsize=(11, 4.5))
ax1 = fig.add_subplot(121, projection="3d")
viz.plot_field_on_points(band_points, probe, title="probe  sin(ωx)sin(ωy)sin(ωz)", ax=ax1)
ax2 = fig.add_subplot(122, projection="3d")
viz.plot_field_on_points(band_points, probe_smoothed, title="after neural_extension ", ax=ax2)
plt.show()

In [ ]:
viz.plot_band_slice_colored(
band_pts=band_points,
values_before=probe,
values_after=probe_smoothed,
axis=2,
level=float(np.median(band_points[:, 2])),
tol=delta_x * 0.9, # pick up ~3 layers of the band in the slice
suptitle="Normal variation before vs after neural_extension",
)
plt.show()

### 2.5 Projecting a band field onto the surface (RBF)

`precompute_rbf_data` caches, per surface point, the $k=8$ nearest band
points, a Cholesky factor of their Gaussian-kernel matrix, and the kernel
vector to that surface point. Then `interpolate_from_precomputed` is just
a couple of small linear solves.

We do this once and reuse it for every PDE solve.

In [ ]:
neighbors_indices, factors, phi_vecs = precompute_rbf_data(
    band_points, Tree_band_points, surface_points, k=8, epsilon=1.0)
print(f"RBF interpolant cached for {surface_points.shape[0]} surface points")

In [ ]:
probe_on_surface = interpolate_from_precomputed(
    probe_smoothed, neighbors_indices, factors, phi_vecs, clipping=True)

fig = plt.figure(figsize=(5, 4.5))
ax = fig.add_subplot(111, projection="3d")
viz.plot_field_on_points(surface_points, probe_on_surface,
                         title="probe (after neural smoothing) on the surface",
                         ax=ax)
plt.show()

## 3. Poisson equation on the Apple

We solve a **Poisson problem** on the surface $S$ (the Apple). The equation we
actually solve is

$$
\boxed{\;\Delta_{S}\, u(x,y,z) \;=\; f(x,y,z)\;}
\qquad\text{with}\qquad
f(x,y,z) \;=\; \sin(\omega x)\,\sin(\omega y)\,\sin(\omega z),\;\;\omega = 10.
$$

In code this becomes

$$
L \, u_{\text{band}} \;=\; \delta x^{2}\, f_{\text{extended}},
$$
Writing the discrete band Laplacian as $L$ (without dividing grid spacing for sparse solvers).

Because $\Delta$ has a one-dimensional kernel (constants) on a closed surface, we remove that ambiguity by enforcing $\int_S f = 0$ and
$\int_S u = 0$, both done by a simple mean-subtraction below.

### 3.1 Assemble the band Laplacian

In [ ]:
Lap, denom = Laplacian_matrix(band_points, Tree_band_points, delta_x)
print(f"L: shape {Lap.shape},  nnz {Lap.nnz},  denom = δx² = {denom:.4e}")
viz.plot_laplacian_spy(Lap)
plt.show()

### 3.2 Extend the right-hand side `f`

We use `neural_extension` on `f` first — this projects the raw 3-D field which is not normal consistent onto the normal consistent space of the surface, which dramatically improves the solution near $S$.

In [ ]:
f_raw = (np.sin(omega*band_points[:,0])
        * np.sin(omega*band_points[:,1])
        * np.sin(omega*band_points[:,2]))
f_raw -= f_raw.mean()
f_smoothed = neural_extension(neural_weights, f_raw,
                        all_local_band_indexes,
                        all_distances_to_central)[0]

fig = plt.figure(figsize=(11, 4.5))
ax1 = fig.add_subplot(121, projection="3d")
viz.plot_field_on_points(band_points, f_raw, title="raw f", ax=ax1)
ax2 = fig.add_subplot(122, projection="3d")
viz.plot_field_on_points(band_points, f_smoothed,
                         title="f after neural_extension", ax=ax2)
plt.show()

### 3.3 Linear solve on the band

We assembled $L$ without the $\delta x^2$ factor; multiply the rhs by it.

In [ ]:
rhs = f_smoothed * denom
u_band = spsolve(Lap, rhs)
u_band -= u_band.mean()

fig = plt.figure(figsize=(5, 4.5))
ax = fig.add_subplot(111, projection="3d")
viz.plot_field_on_points(band_points, u_band, title="Poisson solution on the band", ax=ax)
plt.show()

### 3.4 Project onto the surface

In [ ]:
u_surface_poisson = interpolate_from_precomputed(u_band, neighbors_indices, factors, phi_vecs, clipping=True)

fig = plt.figure(figsize=(5, 4.5))
ax = fig.add_subplot(111, projection="3d")
viz.plot_field_on_points(surface_points, u_surface_poisson, title="Poisson solution on the Apple", ax=ax)
plt.show()

np.save("poisson_solution.npy", u_surface_poisson)
print("saved poisson_solution.npy")

## 4. Heat equation on the Apple

We time-step the **heat equation** on the surface,

$$
\boxed{\;\frac{\partial u}{\partial t}(x, t) \;=\; \Delta_S\, u(x, t)\;}
\qquad x \in S,\;\; t \in [0, T],
$$

with initial condition
$u(x, 0) = \sin(\omega x)\,\sin(\omega y)\,\sin(\omega z)$, $\omega = 10$.

The discrete scheme is forward Euler on the band followed by one `neural_extension`, so that it keeps $u$ (approximately) constant along the normal direction.

$$
u^{\,n+1}_{\text{band}} \;=\; \texttt{neural\_extension}\!\left(\,
u^{\,n}_{\text{band}} + \Delta t\,\Delta^{\text{band}}_h\, u^{\,n}_{\text{band}}\,\right).
$$

In [ ]:
# Same Laplacian as before, but here we divide by δx² (heat equation needs the true Δ)
Lap_heat, _ = Laplacian_matrix(band_points, Tree_band_points, delta_x)
Lap_heat = Lap_heat / denom

dt = 0.1 * delta_x**2
nsteps  = 20
u0 = (np.sin(omega*band_points[:,0]) * np.sin(omega*band_points[:,1]) * np.sin(omega*band_points[:,2]))
u0 = neural_extension(neural_weights, u0, all_local_band_indexes, all_distances_to_central)[0]

U_over_t = np.zeros((nsteps + 1, u0.shape[0]))
U_over_t[0] = u0
u_band   = u0.copy()

for t in tqdm(range(nsteps), desc="heat steps"):
    u_band_new = u_band + dt * (Lap_heat @ u_band)
    u_band = neural_extension(neural_weights, u_band_new, all_local_band_indexes, all_distances_to_central, temperature=0.0423)[0]
    U_over_t[t + 1] = u_band

### 4.1 Snapshots on the band

In [ ]:
frames = [0, 1, 2, 5, 10, nsteps]
viz.snapshot_grid(band_points, U_over_t, frames,
                  suptitle="Heat evolution on the band")
plt.show()

### 4.2 Animate on the surface

We interpolate $u^n_{\text{band}}$ onto the surface for every saved time
step and build an interactive 3-D animation. If the plot below is empty
or black, see the **Plotly inline renderer** note at the top of the
notebook &mdash; or just open the auto-saved `heat_diffusion.html` in a
browser.

In [ ]:
surface_solutions = np.zeros((nsteps + 1, surface_points.shape[0]))
for t in tqdm(range(nsteps + 1), desc="RBF → surface"):
    surface_solutions[t] = interpolate_from_precomputed(
        U_over_t[t], neighbors_indices, factors, phi_vecs, clipping=True)

# plot_3d_point_cloud now ALSO returns the figure so we can display it
# inline; the HTML file is still written as a robust fallback.
fig = plot_3d_point_cloud(
    surface_points, surface_solutions,
    title="Heat diffusion on the Apple (neural CPM)",
    output_file="heat_diffusion.html",
    frame_duration=120,
    colorscale="Magma",
)
fig    # last expression -> rendered inline by the Plotly mime renderer

## 5. Wrap-up

Things to try:

* Change `omega` (RHS frequency) and watch the Poisson solution.
* Change `dist_to_surface` / `delta_x` to see the band density change. Note
  that `local_size = 400` must remain compatible — see the warning printed
  by `define_band_points`.
* Increase the number of rotations in `get_rotations_around_axe` for extra
  robustness (costs more CPU).
